
# Plug flow reactor with surface chemistry

This example simulates the partial oxidation of methane over a platinum catalyst in a
packed bed reactor. This example solves the DAE system directly, using the `FlowReactor`
class and the SUNDIALS IDA solver, in contrast to the approximation as a chain of
steady-state WSRs used in :doc:`surf_pfr_chain.py <surf_pfr_chain>`.

Requires: cantera >= 3.2.0

.. tags:: Python, catalysis, reactor network, surface chemistry, plug flow reactor,
          packed bed reactor


In [1]:
import csv

import cantera as ct

# unit conversion factors to SI
cm = 0.01
minute = 60.0

## Input Parameters



In [2]:
tc = 800.0  # Temperature in Celsius
length = 0.3 * cm  # Catalyst bed length
area = 1.0 * cm**2  # Catalyst bed area
cat_area_per_vol = 1000.0 / cm  # Catalyst particle surface area per unit volume
velocity = 40.0 * cm / minute  # gas velocity
porosity = 0.3  # Catalyst bed porosity

# input file containing the surface reaction mechanism
yaml_file = 'methane_pox_on_pt.yaml'

output_filename = 'surf_pfr2_output.csv'

In [3]:
print(ct.__version__)
print([name for name in dir(ct) if "Surface" in name])

3.2.0
['ReactingSurface1D', 'ReactorSurface', 'Surface1D']


In [4]:
t = tc + 273.15  # convert to Kelvin

# import the model and set the initial conditions
surf = ct.Interface(yaml_file, 'Pt_surf')
surf.TP = t, ct.one_atm
gas = surf.adjacent['gas']
gas.TPX = t, ct.one_atm, 'CH4:1, O2:1.5, AR:0.1'

mass_flow_rate = velocity * gas.density * area * porosity

# create a new reactor
r = ct.FlowReactor(gas, clone=True)
r.area = area
r.surface_area_to_volume_ratio = cat_area_per_vol * porosity
r.mass_flow_rate = mass_flow_rate
r.energy_enabled = False

# Add the reacting surface to the reactor
rsurf = ct.ReactorSurface(surf, r, clone=True)

sim = ct.ReactorNet([r])

output_data = []
n = 0
print('    distance       X_CH4        X_H2        X_CO')
print('  {:10f}  {:10f}  {:10f}  {:10f}'.format(
      0, *r.phase['CH4', 'H2', 'CO'].X))

while sim.distance < length:
    dist = sim.distance * 1e3  # convert to mm
    sim.step()

    if n % 100 == 0 or (dist > 1 and n % 10 == 0):
        print('  {:10f}  {:10f}  {:10f}  {:10f}'.format(
              dist, *r.phase['CH4', 'H2', 'CO'].X))
    n += 1

    # write the gas mole fractions and surface coverages vs. distance
    output_data.append(
        [dist, r.T - 273.15, r.phase.P / ct.one_atm]
        + list(r.phase.X)  # use r.phase.X not gas.X
        + list(rsurf.phase.coverages)  # use rsurf.phase.coverages not surf.coverages
    )

with open(output_filename, 'w', newline="") as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Distance (mm)', 'T (C)', 'P (atm)'] +
                    gas.species_names + surf.species_names)
    writer.writerows(output_data)

print("Results saved to '{0}'".format(output_filename))

    distance       X_CH4        X_H2        X_CO
    0.000000    0.384615    0.000000    0.000000
    0.000000    0.384615    0.000000    0.000000
    0.000001    0.381759    0.001840    0.001738
    0.000002    0.376539    0.004524    0.004479
    0.000005    0.366192    0.008325    0.008692
    0.000012    0.331068    0.013617    0.015671
    0.000025    0.267186    0.011986    0.015957
    0.000044    0.207430    0.007589    0.011266
    0.000069    0.155912    0.004594    0.007920
    0.000078    0.139239    0.003884    0.007706
    0.000081    0.134075    0.003728    0.007991
    0.000082    0.132085    0.003717    0.008242
    0.000083    0.131629    0.003732    0.008334
    0.000083    0.131387    0.003751    0.008398
    0.000083    0.130985    0.003789    0.008519
    0.000083    0.130471    0.003835    0.008681
    0.000084    0.129020    0.003915    0.009135
    0.000085    0.126405    0.003882    0.009888
    0.000087    0.122551    0.003566    0.010784
    0.000090    0.11